# 1. Exploratory analysis

This notebook rebuilds the core analysis: loading and validating the official NPS 2025 package, top and bottom reporting units, annual trends, designations, seasonal patterns, correlations, and DuckDB SQL. The study population is NPS units reporting visitor-use statistics, not all parks visited by the public.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.analysis import load_recreation_visits, load_park_metadata, annual_visits, add_metadata, top_bottom, duckdb_annual_summary
raw = ROOT / 'data' / 'raw'
visits = load_recreation_visits(raw / 'Main_Data.csv')
metadata = load_park_metadata(raw / 'nps_parks_api.json')
annual = add_metadata(annual_visits(visits), metadata)
print(visits.shape, visits.UnitCode.nunique(), visits.Year.min(), visits.Year.max())

## Top and bottom 10
The question is which reporting units have the largest and smallest reported recreation-visit totals. Low values can reflect reporting practices as well as low use.

In [ ]:
latest_year = int(annual.Year.max())
top, bottom = top_bottom(annual, latest_year)
display(top[['ParkName', 'Designation', 'RecreationVisits']])
display(bottom[['ParkName', 'Designation', 'RecreationVisits']])

## Annual trend and 2020
Annual totals show volume, not unique people or crowding. The 2020 break is descriptive and should not be assigned a causal explanation from this dataset alone.

In [ ]:
annual_total = annual.groupby('Year', as_index=False).RecreationVisits.sum()
display(annual_total[annual_total.Year.isin([2019, 2020, 2021, 2024, 2025])])
ax = annual_total.plot(x='Year', y='RecreationVisits', figsize=(12, 5), marker='o', legend=False)
ax.set(title='NPS recreation visits over time', ylabel='Recreation visits')
plt.show()

## Seasonal pattern and DuckDB
Aggregate monthly volume identifies when visits are reported most often across the entire period. It is not a park-level crowding rate.

In [ ]:
monthly = visits.groupby('Month', as_index=False).Value.sum().rename(columns={'Value': 'RecreationVisits'})
display(monthly.sort_values('RecreationVisits', ascending=False))
display(duckdb_annual_summary(annual).head())

## Interpretation
The NPS series recovered substantially after 2020 and reached 323.0 million reported recreation visits in 2025. Rankings vary by year, and designation categories are not interchangeable: National Park units are one subset of the National Park System. Acreage, unique visitors, and direct crowding measures are not available in the verified inputs.